# Kaggle LLM experiment: frozen design and selection

Stage 1 defines the experiment. This notebook makes **no paid calls**, does not construct an API client, and does not run new evaluation policies. Planned episode counts are not research results. Stage 2 now includes the unpaid shared-runner prerequisites below. Genuine validation episodes remain blocked by unresolved gateway usage; primary evaluation has not started.

In VS Code select the project `.venv` kernel and run cells in order. Read `docs/RESEARCH_DESIGN.md` for the full protocol.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/study.json').is_file())
assert Path(sys.prefix).resolve() == (ROOT / '.venv').resolve(), 'Select the project .venv kernel.'
sys.path.insert(0, str(ROOT))
from src.data.kaggle import load_sales
from src.research.design import training_selection, validate_design
study = json.loads((ROOT / 'configs/study.json').read_text(encoding='utf-8'))
print('Study:', study['study_id'])
print('Design status:', study['design_status'])
print('Paid calls authorized:', study['spending']['paid_calls_authorized'])

Study: business_shocks_v1
Design status: frozen_before_paid_pilots
Paid calls authorized: False


## 1. Reproduce training-only selection

Use sample demand CV on training dates, then ranks `floor(N*q)` at q = 0.25, 0.50 and 0.75, with store/product tie-breakers. The table contains training aggregates only; no primary-window policy outcomes are generated or used.

In [2]:
raw = load_sales(ROOT / study['dataset']['path'])
selection_info, selected = training_selection(raw, study['kaggle']['splits']['train_fraction'], study['kaggle']['selection']['quantiles'])
print('Training boundary:', selection_info)
display(pd.DataFrame(selected))

Training boundary: {'train_end': '2023-04-01', 'training_dates': 456, 'eligible_series': 100}


,store_id,product_id,quantile,rank_zero_based,training_mean_demand,training_cv
0,S001,P0001,0.25,25,91.662281,0.382527
1,S004,P0006,0.50,50,122.736842,0.401812
2,S003,P0016,0.75,75,106.949561,0.434536


## 2. Inspect chronology and the prior-use limitation

The existing September 1?21 pilot remains exploratory. The user is unsure whether later outcomes previously informed tuning. The primary window is fixed by this protocol and is **not claimed to be pristine unseen data**. Prompt development is restricted to validation dates. Knowing the date boundaries for analysis does not make the scenario schedule a policy observation.

In [3]:
display(pd.DataFrame(study['kaggle']['windows']).T)
print('Prior manual outcome use:', study['kaggle']['windows']['primary_evaluation']['prior_manual_outcome_use'])
print('Pristine unseen claim allowed:', study['kaggle']['windows']['primary_evaluation']['pristine_unseen_claim_allowed'])

,start,end,purpose,prior_manual_outcome_use,pristine_unseen_claim_allowed,enabled
validation_pilot,2023-04-02,2023-04-22,prompt_and_integration_development,NaN,NaN,NaN
exploratory_existing,2023-09-01,2023-09-21,previously_inspected_baseline_pilot,NaN,NaN,NaN
primary_evaluation,2023-09-22,2023-10-12,fixed_descriptive_comparison_no_tuning,uncertain_user_report_2026-10-17,False,NaN
optional_replication,2023-10-13,2023-11-02,NaN,NaN,NaN,False


Prior manual outcome use: uncertain_user_report_2026-10-17
Pristine unseen claim allowed: False


## 3. Inspect the planned comparison matrix

Three separate single-SKU blocks use four matched scenarios and three policies. The LLM operational policy includes explicit adaptive fallback after one failed repair. Its fallback days remain in the primary results. There are 252 primary LLM decision opportunities; no spending limit is approved by these counts.

In [4]:
coverage = pd.read_csv(ROOT / 'docs/results/planned_coverage.csv')
plan = pd.read_csv(ROOT / 'docs/results/kaggle_episode_plan.csv')
print('Planned native task IDs:', len(coverage.loc[coverage['track'] == 'native']))
print('Kaggle counts:', study['kaggle']['expected_counts'])
assert len(plan) == 36 and plan['episode_id'].is_unique
assert (coverage['completed_units'] == 0).all()
display(plan.groupby(['store_id', 'product_id', 'policy']).size().rename('planned_scenarios').reset_index())
display(coverage.loc[coverage['track'] == 'native', ['unit_id', 'planned_units', 'horizon', 'classification', 'status']])

Planned native task IDs: 19
Kaggle counts: {'matched_blocks': 3, 'episodes': 36, 'policy_days': 756, 'llm_episodes': 12, 'llm_primary_decisions': 252, 'llm_max_attempts_with_repairs': 504}


,store_id,product_id,policy,planned_scenarios
0,S001,P0001,adaptive,4
1,S001,P0001,fixed,4
2,S001,P0001,llm_with_fallback,4
3,S003,P0016,adaptive,4
4,S003,P0016,fixed,4
5,S003,P0016,llm_with_fallback,4
6,S004,P0006,adaptive,4
7,S004,P0006,fixed,4
8,S004,P0006,llm_with_fallback,4


,unit_id,planned_units,horizon,classification,status
0,retail:dynamic_middle,1,180.0,native_logic_compatibility_model_substitution,planned
1,retail:dynamic_hard,1,180.0,native_logic_compatibility_model_substitution,planned
2,retail:still_middle,1,180.0,native_logic_compatibility_model_substitution,planned
3,retail:still_hard,1,180.0,native_logic_compatibility_model_substitution,planned
4,firm:FinCode,7,NaN,native_logic_model_substitution,planned
5,firm:CodeFinQA,50,NaN,native_logic_model_substitution,planned
6,firm:CodeTAT-QA,50,NaN,native_logic_model_substitution,planned
7,firm:SEC-NUM,50,NaN,native_logic_model_substitution,planned
8,firm:TAT-QA,50,NaN,native_logic_model_substitution,planned
9,firm:ConvFinQA,50,NaN,native_logic_model_substitution,planned


## 4. Validate the freeze against real inputs

The checker verifies hashes, source pins, exact native case selection, native horizons, train-only selection (including future-value perturbation), chronology and matched matrix counts. It reads inputs but never runs policies or LLM calls. A failed hash is a reason to inspect the change, not silently regenerate the freeze.

In [5]:
verification = validate_design(ROOT)
assert verification['status'] == 'passed' and verification['paid_api_calls'] == 0
print('Checks passed:', len(verification['checks']))
display(pd.DataFrame(verification['checks']))

Checks passed: 31


,check,status
0,frozen_text_artifacts,passed
1,frozen_native_loader_text,passed
2,pinned_native_sources,passed
3,raw_dataset_checksum,passed
4,dataset_dimensions,passed
5,training_boundary,passed
6,training_only_series_ids,passed
7,training_calibration_values,passed
8,future_values_do_not_change_selection,passed
9,chronological_split_dates,passed


## 5. Interpretation and next work

Primary endpoint: episode contribution versus the adaptive rule within each shock and selected series. Report control-adjusted effects, service, inventory, pricing, recovery, fallback and API cost too. The unit is a matched series-window block; daily rows are correlated repeated measurements. Three selected blocks support descriptive findings, not broad significance or causal claims.

Next implement bounded paid pilots, complete logging and the budget guard in Stage 2, and the shared Kaggle runner/LLM policy in Stage 4. The current data preparer still defaults to the first test window; explicit validation/primary-window support is not yet implemented.

Inspection checklist: selected IDs match the protocol; all statuses remain planned; prior-use uncertainty is visible; no spending authorization is inferred. Figures and result tables will be generated only after actual runs exist.

## Offline LLM-policy implementation checkpoint

The frozen prompt and schema now feed `src/agents/llm.py`. Complete-portfolio validation runs before mutation, one invalid-response repair is allowed, and a second invalid response uses an explicitly logged adaptive fallback. Transport/budget failure stops the episode. `src/experiments/kaggle.py` supplies the same loop for rule and LLM policies; offline tests verify legacy baseline equivalence, privacy, atomicity and full scripted episodes.

The following cell only prints the fixed validation pilot plan: one training-selected series, control and delivery disruption, 21 days each. It does not import an API client or execute any policy. The genuine pilot and the Stage 4 research gate remain incomplete. Use notebook 04 for saved native pilot costs.

In [6]:
import subprocess
import sys
pilot_plan = json.loads(subprocess.check_output([sys.executable, str(ROOT / 'scripts/run_kaggle_experiment.py'),
    '--config', str(ROOT / 'configs/pilots/kaggle.json'), '--dry-run'], cwd=ROOT, text=True))
assert pilot_plan['primary_llm_decisions'] == 42
assert not pilot_plan['paid_calls_authorized_by_plan']
pilot_plan

{'study_id': 'business_shocks_v1',
 'episodes': [{'series': {'store_id': 'S001',
    'product_id': 'P0001',
    'quantile': 0.25,
    'rank_zero_based': 25,
    'training_mean_demand': 91.662280701754,
    'training_cv': 0.382526709661},
   'scenario': 'control',
   'policy': 'llm_with_fallback',
   'window': 'validation_pilot',
   'purpose': 'pilot'},
  {'series': {'store_id': 'S001',
    'product_id': 'P0001',
    'quantile': 0.25,
    'rank_zero_based': 25,
    'training_mean_demand': 91.662280701754,
    'training_cv': 0.382526709661},
   'scenario': 'delivery_disruption',
   'policy': 'llm_with_fallback',
   'window': 'validation_pilot',
   'purpose': 'pilot'}],
 'horizon': 21,
 'primary_llm_decisions': 42,
 'paid_calls_authorized_by_plan': False}